# Demo 07a — Decision Boundaries

**Class 07 · E1, E1b, E1c** (slides: `slides/07_ensembles.md`)

Every model of Classes 03-06 draws a boundary. Here they all learn the **same task**, so that the boundaries can be compared, and then the ensembles of this class are added to the gallery.

* **E1 — Classification gallery:** ten models on two noisy moons ($n = 300$, noise 0.3), a 70/30 stratified split; the **probability of class 1** is the colour (blue to orange), the **0.5 contour** is the decision boundary, **filled points** are training points, **open points** are test points; under each panel the **test accuracy** and the **test log loss**. Then five seeds on five sets, and the same models seen on four resamples of the training set.
* **E1b — Regression twin:** nine regressors on a noisy sine; they are trained on $x \in [-3, 3]$ and queried on $[-4.5, 4.5]$, so the plot shows what each model does *outside* the data.
* **E1c — Disagreement:** on how many test points do two models differ? A combination can only help if the members make *different* mistakes.

Convention (Class 01): in a 2-class problem the positive class is 1 (orange). All data are generated with fixed seeds (no download). Every cell that prints a number of a slide starts with `slide: <frame title>`.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.datasets import make_moons
from sklearn.neighbors import KNeighborsClassifier
from sklearn.utils import resample
from threadpoolctl import threadpool_limits

from faa import boundaries, data, galleries, metrics, plotlib, synthetic

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
pl.Config.set_fmt_str_lengths(40)


plotlib.setup(module="07")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0
N_JOBS = 8


def slide(title: str) -> None:
    """Every cell that prints a number of a slide starts with this line: the frame title of the deck."""
    print(f"--- slide: {title} ---")


def show(frame: pl.DataFrame, digits: int = 3) -> None:
    """Print a polars table with rounded floats."""
    print(frame.with_columns(pl.col(pl.Float64).round(digits)))


def fit_gallery(X_tr, y_tr, seed):
    """Fit the ten classifiers of the gallery on one training set."""
    with threadpool_limits(1):
        return {name: m.fit(X_tr, y_tr) for name, m in galleries.gallery_models(seed).items()}

## E1 — Ten models, one task

Moons, $\sigma = 0.3$: two interleaved half circles with a lot of overlap, so no model can reach 1.00 on the test set. The ten models, in the order of the figure:

* **single models of Classes 03-06:** logistic regression (a straight line), Gaussian naive Bayes, SVM with an RBF kernel, MLP (2 x 32 units), 1-NN, 15-NN, an unpruned decision tree;
* **ensembles of this class:** a random forest (200 trees), gradient boosting (100 trees of depth 3);
* **vote:** the *soft vote* (mean of the predicted probabilities) of the six single models, 1-NN excluded.

The log loss clips the probabilities at $10^{-3}$ and $1 - 10^{-3}$: 1-NN and a tree give 0 or 1, and one wrong 0/1 answer would be an infinite loss.

In [ ]:
slide("The Moons: One Running Example")
sets = synthetic.class_sets(SEED)
X, y = sets["moons"]
X_tr, X_te, y_tr, y_te = data.split(X, y, SEED)
print(
    f"moons: n = {len(y)}, noise 0.3; train {len(y_tr)} (class 1: {int(y_tr.sum())}), test {len(y_te)} (class 1: {int(y_te.sum())})"
)
print(
    "the other sets (n = 300 each):",
    ", ".join(f"{k} (class 1: {int(v[1].sum())})" for k, v in sets.items() if k != "moons"),
)

In [ ]:
slide("One Model Is a Guess: the Gallery on the Moons")
models = fit_gallery(X_tr, y_tr, SEED)
rows = []
for name, m in models.items():
    acc, ll = metrics.scores(m, X_te, y_te)
    rows.append((name, float(m.score(X_tr, y_tr)), acc, ll))
table_moons = pl.DataFrame(rows, schema=["model", "train acc", "test acc", "test log loss"], orient="row")
show(table_moons)
best = table_moons.sort("test acc", descending=True)
print(
    f"best single test accuracy: {best['model'][0]} {best['test acc'][0]:.3f}; worst: {best['model'][-1]} {best['test acc'][-1]:.3f}"
)
print(
    f"spread of the ten test accuracies: {table_moons['test acc'].min():.3f} to {table_moons['test acc'].max():.3f} ({len(y_te)} test points: one point = {1 / len(y_te):.3f})"
)

In [ ]:
slide("The Ceiling of the Moons")
Xbig, ybig = make_moons(40_000, noise=0.3, random_state=1)
Xhold, yhold = make_moons(20_000, noise=0.3, random_state=2)
ceiling = {k: float(KNeighborsClassifier(k).fit(Xbig, ybig).score(Xhold, yhold)) for k in (50, 150, 400)}
print(
    "test accuracy of k-NN fitted on 40 000 moons (k = 50, 150, 400):", ", ".join(f"{v:.3f}" for v in ceiling.values())
)
print(
    f"so the best possible accuracy on the moons with noise 0.3 is about {max(ceiling.values()):.2f}: on our 90 test points the models are all within a few points of the ceiling or far below it"
)

In [ ]:
# plot e1_gallery_moons
lim = boundaries.plane_limits(X)
fg, axs = boundaries.new_fig(2, 5, panel_h=1.02)
for ax, (name, m) in zip(axs.ravel(), models.items()):
    boundaries.boundary_panel(ax, m, X_tr, y_tr, X_te, y_te, title=name, lim=lim)
fg.subplots_adjust(wspace=0.06, hspace=0.42)
plotlib.save(fg, "e1_gallery_moons")

## Does the ranking hold? Five seeds on five sets

One split of 90 test points is one draw (Class 01): one point is 0.011 of accuracy. The table repeats the gallery on **five sets** (moons, overlapping blobs, circles, two clusters per class, XOR) with **five seeds** each, and ranks the models.

What the printed tables show:

* **No model wins everywhere.** The straight line of logistic regression is the right shape for blobs and clusters (0.85 and 0.90) and the wrong one for XOR (0.54) and circles (0.46, worse than a coin); naive Bayes recovers the circles (0.95) because it models the variance of each class, but not XOR.
* **The single tree is the worst of the flexible models** (mean accuracy 0.848, mean log loss 1.05): it fits the training points perfectly and its test accuracy varies most between seeds.
* **The two tree ensembles add about three points to the tree** (forest 0.880, boosting 0.875), but on these small 2-D sets they do **not** beat a well-tuned smooth model (SVM RBF 0.901, MLP 0.900, 15-NN 0.897). The vote of six models, three of them weak, is 0.892.
* The mean **log loss** separates the models more than accuracy does: 1-NN and the tree are confident and wrong (about 1.0), the smooth models and the ensembles give calibrated probabilities (0.26 to 0.35).

The next cells draw the best single model (by mean accuracy) against the forest and boosting on the four other sets.

In [ ]:
slide("Does the Ranking Hold? Five Seeds, Five Sets")


def one_run(set_name, seed):
    """Test accuracy and log loss of the ten models on one set drawn with `seed`."""
    X_, y_ = synthetic.class_sets(seed)[set_name]
    a, b, c, d = data.split(X_, y_, seed)
    fitted = fit_gallery(a, c, seed)
    return [(set_name, seed, name, *metrics.scores(m, b, d)) for name, m in fitted.items()]


runs = Parallel(n_jobs=N_JOBS)(delayed(one_run)(s, seed) for s in sets for seed in range(5))
long = pl.DataFrame([r for run in runs for r in run], schema=["set", "seed", "model", "acc", "ll"], orient="row")
order = list(galleries.gallery_models())
summary = long.group_by("set", "model").agg(
    pl.col("acc").mean().alias("acc"), pl.col("acc").std().alias("sd"), pl.col("ll").mean().alias("ll")
)
wide = (
    summary.with_columns(
        pl.struct("acc", "sd")
        .map_elements(lambda r: f"{r['acc']:.2f}±{r['sd']:.2f}", return_dtype=pl.String)
        .alias("cell")
    )
    .pivot(on="set", index="model", values="cell")
    .select("model", *sets)
)
wide = wide.join(pl.DataFrame({"model": order, "i": range(len(order))}), on="model").sort("i").drop("i")
print("test accuracy, mean ± sd over 5 seeds (70/30 split, n = 300):")
print(wide)
mean_acc = (
    long.group_by("model")
    .agg(pl.col("acc").mean().alias("mean acc"), pl.col("ll").mean().alias("mean ll"))
    .sort("mean acc", descending=True)
)
show(mean_acc)
ranks = (
    long.group_by("set", "seed", "model")
    .agg(pl.col("acc").mean())
    .with_columns(pl.col("acc").rank(method="average", descending=True).over("set", "seed").alias("rank"))
    .group_by("model")
    .agg(pl.col("rank").mean())
    .sort("rank")
)
print("mean rank over the 25 (set, seed) runs (1 = best):")
show(ranks, 2)
single = [m for m in mean_acc["model"].to_list() if m not in ("forest", "boosting", "vote")]
print(
    "best single (non-ensemble) model by mean test accuracy:",
    single[0],
    f"{mean_acc.filter(pl.col('model') == single[0])['mean acc'][0]:.3f}",
)
BEST3 = [single[0], "forest", "boosting"]  # the best single model against the two tree ensembles
print("models drawn on the other sets:", BEST3)

In [ ]:
# plot e1_sets: 0.5 contours of the three best models on the four other sets
fg, axs = plt.subplots(2, 2, figsize=(plotlib.COL, 2.05))
COLS = [BLUE, GREEN, RED]
for ax, set_name in zip(axs.ravel(), [k for k in sets if k != "moons"]):
    X_, y_ = sets[set_name]
    a, b, c, d = data.split(X_, y_, SEED)
    fitted = fit_gallery(a, c, SEED)
    lim_ = boundaries.plane_limits(X_)
    for cl in (0, 1):
        ax.scatter(*a[c == cl].T, s=3.5, color=boundaries.CLASS_COLOURS[cl], edgecolor="white", linewidth=0.2, zorder=3)
    for name, col, ls in zip(BEST3, COLS, ("-", "--", ":")):
        boundaries.contour_line(ax, fitted[name], lim_, col, ls=ls, lw=1.0)
    boundaries.clean_axes(ax, lim_)
    ax.set_title(set_name, fontsize=6.5, pad=2)
handles = [plt.Line2D([], [], color=col, ls=ls, lw=1.0, label=n) for n, col, ls in zip(BEST3, COLS, ("-", "--", ":"))]
fg.legend(
    handles=handles, loc="lower center", ncols=3, fontsize=6, handlelength=1.6, columnspacing=0.8, borderaxespad=0
)
fg.subplots_adjust(wspace=0.08, hspace=0.25, bottom=0.12)
plotlib.save(fg, "e1_sets")

In [ ]:
# plot e1_gallery_sets: the three best models on the four other sets, with their test numbers
other = [k for k in sets if k != "moons"]
fg, axs = boundaries.new_fig(3, 4, panel_h=0.98)
for j, set_name in enumerate(other):
    X_, y_ = sets[set_name]
    a, b, c, d = data.split(X_, y_, SEED)
    fitted = fit_gallery(a, c, SEED)
    lim_ = boundaries.plane_limits(X_)
    for i, name in enumerate(BEST3):
        boundaries.boundary_panel(
            axs[i, j], fitted[name], a, c, b, d, title=f"{set_name}: {name}", lim=lim_, fontsize=5.6
        )
fg.subplots_adjust(wspace=0.06, hspace=0.45)
plotlib.save(fg, "e1_gallery_sets")

## Bias and variance in the gallery (Classes 01 and 06)

The same 210 training points, resampled with replacement four times (a bootstrap, see E3), give four models of each family. Look at the **row**, not at one panel:

* **logistic regression:** four almost identical lines; the model is too simple for the moons (accuracy 0.77, sd 0.014): **high bias, low variance**;
* **tree:** four different sets of boxes; one point moves a cut (accuracy 0.89, sd 0.037): **low bias, high variance**;
* **15-NN:** a curved boundary that also moves with the sample.

An average of the four trees would move less than any one of them: that is the idea of Class 07.

In [ ]:
slide("Bias and Variance Seen in the Gallery")
rng = np.random.default_rng(SEED)
FOUR = [resample(np.arange(len(y_tr)), random_state=int(s)) for s in rng.integers(0, 10_000, 4)]
ROWS = ["logistic", "tree", "15-NN"]
base = galleries.gallery_models(SEED)
fig_rows, spread = [], []
fg, axs = boundaries.new_fig(3, 4, panel_h=0.95)
for i, name in enumerate(ROWS):
    accs = []
    for j, idx in enumerate(FOUR):
        with threadpool_limits(1):
            m = clone(base[name]).fit(X_tr[idx], y_tr[idx])
        accs.append(
            boundaries.boundary_panel(
                axs[i, j],
                m,
                X_tr[idx],
                y_tr[idx],
                X_te,
                y_te,
                title=f"resample {j + 1}" if i == 0 else "",
                lim=lim,
                points=False,
            )[0]
        )
        if j == 0:
            axs[i, j].set_ylabel(name, fontsize=6.5, labelpad=2)
    spread.append((name, float(np.mean(accs)), float(np.std(accs, ddof=1))))
fg.subplots_adjust(wspace=0.06, hspace=0.42)
print("test accuracy of the model fitted on 4 bootstrap resamples of the same 210 training points:")
show(pl.DataFrame(spread, schema=["model", "mean acc", "sd of acc"], orient="row"))
plotlib.save(fg, "e1_variance")

### Failure case: more noise (no slide)

The same moons with the noise $\sigma$ from 0.1 to 0.5 (mean test accuracy of 5 seeds). At $\sigma = 0.1$ the flexible models are near-perfect (1-NN 0.996) and the line is the worst (0.869). At $\sigma = 0.5$ the order has turned: **1-NN (0.727) and the tree (0.751) are below the straight line (0.807)** because they fit the noise; the forest (0.800) and boosting (0.793) lose much less than the tree they are built from.

In [ ]:
slide("How a Boundary Reacts to the Noise Level")
noise_rows = []
KEEP = ("logistic", "1-NN", "tree", "forest", "boosting")


def noise_run(noise, seed):
    Xn, yn = make_moons(300, noise=noise, random_state=seed)
    a, b, c, d = data.split(Xn, yn, seed)
    with threadpool_limits(1):
        models_n = {n: m.fit(a, c) for n, m in galleries.gallery_models(seed).items() if n in KEEP}
    return [metrics.scores(m, b, d)[0] for m in models_n.values()]


NOISES = (0.1, 0.2, 0.3, 0.4, 0.5)
res = Parallel(n_jobs=N_JOBS)(delayed(noise_run)(nz, sd) for nz in NOISES for sd in range(5))
for i, nz in enumerate(NOISES):
    noise_rows.append((nz, *np.mean(res[5 * i : 5 * i + 5], axis=0).tolist()))
show(pl.DataFrame(noise_rows, schema=["noise", "logistic", "1-NN", "tree", "forest", "boosting"], orient="row"))

### Real data in two dimensions (E1d)

Generated sets show what a boundary *can* look like; real data show what it *does* look like. Two real tables, each reduced to **two standardised features** so that the boundary can be drawn (the models of the rest of the notebook use all features):

* **breast cancer** (`worst_concave_points`, `worst_texture`; class 1 = malignant): two informative features, almost separable, so every model draws a similar boundary;
* **California map** (`Longitude`, `Latitude`; class 1 = house value above the median): the classes follow the coast, a boundary that no line can draw, so the models that bend (KNN, trees, forests, boosting) separate from the linear one (1000 districts).

In [ ]:
slide("Real Data in Two Dimensions")
REAL = {"breast cancer": "breast", "California map": "california_map"}
SHOW = ("logistic", "SVM RBF", "15-NN", "tree", "forest")
real_rows = []
fg, axs = boundaries.new_fig(2, len(SHOW), panel_h=1.12)
for r, (title, key) in enumerate(REAL.items()):
    Xr, yr, cols = data.real_2d(key, n=1000, seed=SEED)
    a, b, c, d = data.split(Xr, yr, SEED)
    lim = boundaries.plane_limits(Xr)
    axs[r][0].set_ylabel(title, fontsize=6.5)
    zoo = galleries.gallery_models(SEED)
    for ax, name in zip(axs[r], SHOW, strict=True):
        m = zoo[name].fit(a, c)
        acc, ll = boundaries.boundary_panel(ax, m, a, c, b, d, title=name, lim=lim)
        real_rows.append((title, name, round(acc, 3), round(ll, 3)))
print(pl.DataFrame(real_rows, schema=["data", "model", "test accuracy", "test log loss"], orient="row"))
print("features:", {t: data.real_2d(k, n=1000, seed=SEED)[2] for t, k in REAL.items()}, "; 1000 rows each, 70/30 split")
plotlib.save(fg, "e1_real_boundaries")

## E1b — Regression twin: curves and extrapolation

A noisy sine ($\sigma = 0.3$, 80 training points in $[-3, 3]$) and nine regressors. Under each panel: the RMSE against the noiseless sine **inside** and **outside** the training range (the shaded band). The test points are in $[-4.5, 4.5]$.

* **Inside**, the smooth models (polynomial of degree 5, SVR, MLP, 5-NN, the vote) track the curve (RMSE 0.09 to 0.14); the single tree is a staircase (0.29) and the forest and boosting average many staircases (0.20 and 0.22).
* **Outside**, trees, forests, boosting and KNN are **flat**: they repeat the value of the last leaf (RMSE 0.55 to 0.66, as bad as the straight line); the polynomial explodes (6.8). Nothing here extrapolates the sine: a model cannot know a pattern it never saw.
* On a target with a trend (the **cubic**), the tree models are flat and miss it badly (RMSE outside about 5.3 against 2.1 for the polynomial that has the right shape).

In [ ]:
slide("Regression Twin: Curves and Extrapolation")
Xr_tr, yr_tr, Xr_te, yr_te, f_sine = synthetic.reg_data("sine", SEED)
reg = {}
with threadpool_limits(1):
    for name, m in galleries.reg_models(SEED).items():
        reg[name] = m.fit(Xr_tr, yr_tr)
rows = [(name, *metrics.range_rmse(m, Xr_te, f_sine)) for name, m in reg.items()]
t_sine = pl.DataFrame(rows, schema=["model", "RMSE in range", "RMSE outside"], orient="row")
show(t_sine)
print(
    f"noise sd = {synthetic.TARGETS['sine'][1]}; training range {synthetic.TRAIN_RANGE}, test range {synthetic.TEST_RANGE}; {len(yr_tr)} training points"
)

In [ ]:
# plot e1b_sine
fg, axs = boundaries.new_fig(2, 5, panel_h=0.98)
YL = (-1.8, 1.8)
for ax, (name, m) in zip(axs.ravel(), reg.items()):
    boundaries.curve_panel(ax, m, Xr_tr, yr_tr, Xr_te, yr_te, f_sine, title=name, ylim=YL)
key = axs.ravel()[-1]
key.axis("off")
key.scatter([], [], s=6, color=ORANGE, label="train point")
key.scatter([], [], s=6, facecolor="none", edgecolor=GREY, label="test point")
key.plot([], [], color=boundaries.LINE, ls=":", lw=0.8, label="true curve")
key.plot([], [], color=BLUE, lw=1.0, label="prediction")
key.legend(loc="center", fontsize=5.8, handlelength=1.4, borderaxespad=0)
fg.subplots_adjust(wspace=0.06, hspace=0.42)
plotlib.save(fg, "e1b_sine")

### Three targets, ten noise draws (no slide)

A sine, a cubic (a trend that keeps growing) and a step, each with ten noise draws. The table reports the mean RMSE against the noiseless function inside and outside the training range; the `sd` columns are the spread over the draws, the variance of the model seen as a number.

In [ ]:
slide("Three Targets, Ten Noise Draws")


def reg_run(target, seed):
    a, b, c, d, f = synthetic.reg_data(target, seed)
    with threadpool_limits(1):
        return [
            (target, seed, n, *metrics.range_rmse(m.fit(a, b), c, f)) for n, m in galleries.reg_models(seed).items()
        ]


rr = Parallel(n_jobs=N_JOBS)(delayed(reg_run)(t, s) for t in synthetic.TARGETS for s in range(10))
rl = pl.DataFrame([r for run in rr for r in run], schema=["target", "seed", "model", "in", "out"], orient="row")
agg = rl.group_by("target", "model").agg(
    pl.col("in").mean(), pl.col("in").std().alias("in sd"), pl.col("out").mean(), pl.col("out").std().alias("out sd")
)
order_r = list(galleries.reg_models())
for t in synthetic.TARGETS:
    sub = (
        agg.filter(pl.col("target") == t)
        .join(pl.DataFrame({"model": order_r, "i": range(len(order_r))}), on="model")
        .sort("i")
        .drop("i", "target")
    )
    print(
        f"target {t}: RMSE against the noiseless function, mean over 10 noise draws (standard deviations over the draws in the 'sd' columns)"
    )
    show(sub)

In [ ]:
# plot e1b_extrap: error against x for the models, sine target, 30 noise draws
def predictions(seed):
    a, b, c, d, f = synthetic.reg_data("sine", seed)
    xs = np.linspace(*synthetic.TEST_RANGE, 91)[:, None]
    with threadpool_limits(1):
        return {n: m.fit(a, b).predict(xs) for n, m in galleries.reg_models(seed).items()}, f(xs[:, 0]), xs[:, 0]


pr = Parallel(n_jobs=N_JOBS)(delayed(predictions)(s) for s in range(30))
xs = pr[0][2]
truth = pr[0][1]
fg, ax = plotlib.fig("full", 1.5)
SHOW = {
    "polynomial 5": (RED, "-"),
    "SVR RBF": (SKY, "-"),
    "5-NN": (PURPLE, "-"),
    "tree": (GREEN, "-"),
    "forest": (BLUE, "--"),
    "boosting": (ORANGE, "--"),
}
edges = np.linspace(*synthetic.TEST_RANGE, 10)
mid = (edges[:-1] + edges[1:]) / 2
extrap_rows = []
for name, (col, ls) in SHOW.items():
    err = np.array([(p[0][name] - p[1]) ** 2 for p in pr])  # draws x x-values
    per_x = np.sqrt(err.mean(axis=0))
    binned = [np.sqrt(np.mean(err[:, (xs >= lo) & (xs <= hi)])) for lo, hi in zip(edges[:-1], edges[1:])]
    ax.plot(mid, binned, color=col, ls=ls, marker="o", ms=2.2, label=name)
    extrap_rows.append(
        (
            name,
            float(np.sqrt(err[:, np.abs(xs) <= 3].mean())),
            float(np.sqrt(err[:, np.abs(xs) > 3].mean())),
            float(per_x[-1]),
        )
    )
ax.axvspan(*synthetic.TRAIN_RANGE, color="#EEF1F4", lw=0, zorder=0)
ax.set(
    yscale="log",
    xlabel="$x$ (shaded band: training range)",
    ylabel="RMSE of the prediction (log scale)",
    xlim=synthetic.TEST_RANGE,
)
ax.legend(ncols=6, loc="upper center", bbox_to_anchor=(0.5, 1.22), fontsize=6.3, handlelength=1.3, columnspacing=0.8)
show(pl.DataFrame(extrap_rows, schema=["model", "RMSE inside", "RMSE outside", "RMSE at x = 4.5"], orient="row"))
plotlib.save(fg, "e1b_extrap")

### Real data in one dimension (E1e)

The sine has a known truth; real data have none, only noise. California housing, **median house value against median income** (300 districts): the regressors are trained on the districts up to the 75th percentile of the income and queried on all of them, so the right part of each panel is extrapolation. The label under a panel is the test RMSE against the noisy values; the table adds the RMSE beyond the training incomes. A real target is noisy and capped at 5.0 (the last column of the table is where the models differ most), so a straight line is hard to beat here and a polynomial explodes.

In [ ]:
slide("Real Data: House Value against Income")
Xh_tr, yh_tr, Xh_te, yh_te, rng_tr, rng_te = data.real_1d_regression(SEED)
print(
    f"California housing: {len(yh_tr)} training districts with income <= {rng_tr[1]:.2f}, {len(yh_te)} test districts up to {rng_te[1]:.2f} (units of 10 000 USD)"
)
fg, axs = boundaries.new_fig(2, 5, panel_h=0.98)
rows_h = []
for ax, (name, m) in zip(axs.ravel(), galleries.reg_models(SEED).items(), strict=False):
    m.fit(Xh_tr, yh_tr)
    r_in, _ = boundaries.curve_panel(
        ax, m, Xh_tr, yh_tr, Xh_te, yh_te, None, title=name, ylim=(0, 5.5), train_range=rng_tr, test_range=rng_te
    )
    rows_h.append(
        (
            name,
            round(r_in, 3),
            round(metrics.rmse(m.predict(Xh_te[Xh_te[:, 0] > rng_tr[1]]), yh_te[Xh_te[:, 0] > rng_tr[1]]), 3),
        )
    )
for ax in axs.ravel()[len(rows_h) :]:
    ax.axis("off")
print(pl.DataFrame(rows_h, schema=["model", "test RMSE (all)", "RMSE beyond the training incomes"], orient="row"))
plotlib.save(fg, "e1b_real")

## E1c — The errors of the models are not the same

On the moons test set (90 points), how many points does each pair of models classify differently? A combination can only help when the members **disagree where they are wrong**. The heat map counts the disagreements of the ten models of the gallery.

* 65 of the 90 test points are right for all ten models, 3 are wrong for all ten (nobody can fix them), and **22 are a split decision**: these are the points a combination can win or lose.
* The oracle that picks the best model per point would reach 0.967, the best single model reaches 0.956 (15-NN) and the majority vote of the ten hard predictions 0.900: **diversity is available, but a vote does not collect it for free**; the weak members (logistic, naive Bayes: 0.756) vote too.
* Logistic regression and naive Bayes give the same 90 predictions here (0 disagreements); forest and boosting differ on 2 points, tree and forest on 4. Models of the same family make the same mistakes.

In [ ]:
slide("The Errors of the Models Are Not the Same")
pred = np.array([m.predict(X_te) for m in models.values()])  # 10 x 90
names = list(models)
wrong = pred != y_te
n_wrong = wrong.sum(axis=0)
print(f"{len(y_te)} test points; per point, the number of the 10 models that are wrong:")
show(pl.DataFrame({"models wrong": range(11), "points": np.bincount(n_wrong, minlength=11)}))
print(
    f"all ten right: {int(np.sum(n_wrong == 0))}; all ten wrong: {int(np.sum(n_wrong == 10))}; split decision (1 to 9 wrong): {int(np.sum((n_wrong > 0) & (n_wrong < 10)))}"
)
print(f"oracle accuracy (at least one model right): {np.mean(n_wrong < 10):.3f}")
maj = (pred.mean(axis=0) > 0.5).astype(int)
print(
    f"majority vote of the 10 hard predictions: test accuracy {np.mean(maj == y_te):.3f}; best single model {best['model'][0]} {best['test acc'][0]:.3f}"
)
D = np.array([[int(np.sum(a != b)) for b in pred] for a in pred])
iu = np.triu_indices(10, 1)
print(
    f"disagreement counts between two models: mean {D[iu].mean():.1f}, min {D[iu].min()} ({names[iu[0][D[iu].argmin()]]} vs {names[iu[1][D[iu].argmin()]]}), max {D[iu].max()} ({names[iu[0][D[iu].argmax()]]} vs {names[iu[1][D[iu].argmax()]]})"
)
print(f"disagreement of logistic vs 15-NN: {D[0, 5]} points; tree vs forest: {D[6, 7]}; forest vs boosting: {D[7, 8]}")
short = ["logis.", "NB", "SVM", "MLP", "1-NN", "15-NN", "tree", "forest", "boost.", "vote"]
show(pl.DataFrame(D, schema=short, orient="row").with_columns(pl.Series("model", names)).select("model", *short), 0)

In [ ]:
# plot e1c_disagree
fg, ax = plt.subplots(figsize=(plotlib.COL, 2.15))
mask = np.tril(np.ones((9, 9), dtype=bool), -1)  # the upper triangle of the symmetric matrix, without the diagonal
sns.heatmap(
    D[:-1, 1:],
    mask=mask,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    square=True,
    vmin=0,
    vmax=D[iu].max() * 1.3,
    linewidths=0.4,
    linecolor="white",
    annot_kws={"fontsize": 5.4},
    xticklabels=short[1:],
    yticklabels=short[:-1],
    ax=ax,
)
ax.tick_params(labelsize=5.6, length=0)
plt.setp(ax.get_xticklabels(), rotation=60, ha="right")
ax.grid(False)
ax.set_title(f"test points where two models differ (of {len(y_te)})", fontsize=6.3, pad=2)
plotlib.save(fg, "e1c_disagree")

## Summary

* One task, ten models, ten different boundaries: straight, smooth, axis-parallel boxes, jagged cells. Test accuracy on the moons runs from 0.756 to 0.956.
* The ranking changes with the set; the single tree is the least stable model and the least calibrated one.
* Trees, forests, boosting and k-NN are flat outside the data (E1b).
* The models make **different** mistakes (E1c): that is the raw material of an ensemble. Next: why averaging helps, and how to build the members (`02_bagging_forests.ipynb`).